# Agente de análise de estoque (LangGraph + Ollama)

Fluxo: **pergunta → `interpretar` (LLM) → decisão → `consultar_api` (ferramenta) → `responder`**, ou direto para `responder_direto` quando a API não é necessária.

Pré-requisitos:
- API de estoque no ar em `http://127.0.0.1:8000` (repositório `api-estoque`).
- Ollama rodando com o modelo `qwen2.5:7b` (local e gratuito).
- Kernel do notebook: o Python do `.venv` deste projeto.

## 1. Configuração

In [1]:
import os
import re
from typing import Literal, Optional, TypedDict

import httpx
from dotenv import load_dotenv
from langchain_core.tools import tool
from langchain_ollama import ChatOllama
from langgraph.graph import END, START, StateGraph
from pydantic import BaseModel, Field

load_dotenv()  # lê o .env, se existir (ele não é versionado)
API_URL = os.getenv("API_ESTOQUE_URL", "http://127.0.0.1:8000")
MODELO = os.getenv("OLLAMA_MODEL", "qwen2.5:7b")

# temperature=0 deixa as respostas estáveis; timeout evita ficar preso se o Ollama travar
llm = ChatOllama(model=MODELO, temperature=0, client_kwargs={"timeout": 120})
print(f"Configuração realizada com sucesso (API: {API_URL} | modelo: {MODELO})")

Configuração realizada com sucesso (API: http://127.0.0.1:8000 | modelo: qwen2.5:7b)


## 2. Ferramenta: `consultar_produto`

- **Finalidade:** consultar um produto na API de estoque (`GET /produtos/{codigo}`).
- **Parâmetro:** `codigo` no formato `PRD-XXX`.
- **Retorno:** `{"ok": True, "produto": {...}}` ou `{"ok": False, "erro": ..., "mensagem": ...}`.
- **Falhas tratadas:** código inválido (validado antes da chamada), produto inexistente (404), serviço indisponível (503), API desligada/timeout e resposta com dados inválidos.

In [ ]:
CODIGO_VALIDO = re.compile(r"^PRD-[0-9]{3}$")
CAMPOS_OBRIGATORIOS = {"codigo", "nome", "quantidade_atual", "estoque_minimo"}


@tool
def consultar_produto(codigo: str) -> dict:
    """Consulta um produto na API de estoque pelo código (formato PRD-XXX, ex.: PRD-001).

    Retorna {"ok": True, "produto": {...}} ou {"ok": False, "erro": <identificador>, "mensagem": <texto>}.
    """
    codigo = codigo.strip().upper()
    if not CODIGO_VALIDO.match(codigo):
        return {"ok": False, "erro": "codigo_invalido",
                "mensagem": f"O código '{codigo}' não segue o formato PRD-XXX (ex.: PRD-001)."}

    try:
        r = httpx.get(f"{API_URL}/produtos/{codigo}", timeout=5)
    except httpx.HTTPError:
        return {"ok": False, "erro": "api_inacessivel",
                "mensagem": "Não foi possível conectar à API de estoque. Verifique se ela está no ar."}

    try:
        corpo = r.json()
    except ValueError:
        corpo = None
    if not isinstance(corpo, dict):
        corpo = {}

    if r.status_code != 200:
        return {"ok": False, "erro": corpo.get("erro", f"http_{r.status_code}"),
                "mensagem": corpo.get("mensagem", "Erro inesperado da API.")}

    # Verifica o retorno antes de usar: campos presentes e quantidades inteiras >= 0
    quantidades_ok = all(isinstance(corpo.get(c), int) and corpo[c] >= 0
                         for c in ("quantidade_atual", "estoque_minimo"))
    if not (CAMPOS_OBRIGATORIOS <= corpo.keys() and quantidades_ok):
        return {"ok": False, "erro": "resposta_invalida",
                "mensagem": "A API retornou dados incompletos ou inválidos."}
    return {"ok": True, "produto": corpo}

print("Ferramenta consultar_produto criada com sucesso")

Ferramenta consultar_produto criada com sucesso


## 3. Regra de situação do estoque

Calculada em Python (não pelo LLM), para o resultado ser sempre correto e verificável:

| Condição | Situação |
|---|---|
| `quantidade_atual == 0` | esgotado |
| `0 < quantidade_atual <= estoque_minimo` | baixo (inclui estar exatamente no mínimo) |
| `quantidade_atual > estoque_minimo` | disponível |

In [ ]:
def classificar_situacao(quantidade: int, minimo: int) -> str:
    if quantidade == 0:
        return "esgotado"
    if quantidade <= minimo:
        return "baixo"
    return "disponível"

print("Regra classificar_situacao definida com sucesso")

Regra classificar_situacao definida com sucesso


## 4. Instruções do agente (papel, objetivo, contexto e restrições)

In [4]:
PROMPT_SISTEMA = """Papel: você é um assistente de análise de estoque de uma loja.
Objetivo: informar a disponibilidade e a situação do estoque de produtos.
Contexto: os dados vêm exclusivamente da API de estoque. Cada produto tem código PRD-XXX,
quantidade_atual e estoque_minimo. Situações: esgotado (quantidade 0), baixo (quantidade maior
que 0 e menor ou igual ao mínimo) e disponível (acima do mínimo).
O estoque mínimo é o ponto de reposição: quando a quantidade atual chega ao mínimo ou fica abaixo dele,
o produto precisa ser reposto para não esgotar.
Restrições:
- Use somente os dados fornecidos; nunca invente produtos, códigos ou quantidades.
- Você apenas consulta: não cadastra, altera nem exclui produtos.
- Ao falar de um produto, informe sempre a quantidade atual e o estoque mínimo.
- Responda em português, de forma breve e objetiva."""

PROMPT_CLASSIFICACAO = """Classifique a mensagem do usuário em uma intenção:
- consultar_produto: pede estoque, quantidade, disponibilidade ou situação de um produto e informa um código
  (copie o código exatamente como o usuário escreveu no campo codigo, mesmo que pareça errado).
- conceitual: pergunta geral sobre conceitos de estoque (ex.: o que é estoque mínimo), sem pedir dados de um produto.
- ambigua: quer dados de produto, mas não informou nenhum código.
- fora_do_escopo: qualquer outro assunto, inclusive pedidos para cadastrar, alterar ou excluir produtos.

Exemplos:
- "Qual a situação do PRD-001?" -> consultar_produto, codigo PRD-001
- "Tem o item ABC-1 em estoque?" -> consultar_produto, codigo ABC-1
- "O que significa estoque baixo?" -> conceitual
- "Como está o estoque do mouse?" -> ambigua
- "Cadastre o produto PRD-020" -> fora_do_escopo
- "Qual a capital da França?" -> fora_do_escopo"""

print("Prompts do agente definidos")

Prompts do agente definidos


## 5. Estado compartilhado e nós do grafo

- `interpretar`: o LLM classifica a intenção e extrai o código (saída estruturada).
- `decidir_rota`: **transição condicional** — usa a ferramenta só quando há produto a consultar.
- `consultar_api`: chama a ferramenta e calcula a situação.
- `responder`: o LLM redige a resposta com base **apenas** nos dados da API; em falha, mensagem fixa (sem LLM, para não inventar dados).
- `responder_direto`: perguntas conceituais (LLM, sem API) ou mensagens fixas para ambígua/fora do escopo.

In [5]:
class Decisao(BaseModel):
    intencao: Literal["consultar_produto", "conceitual", "ambigua", "fora_do_escopo"]
    codigo: Optional[str] = Field(None, description="Código do produto citado, exatamente como escrito pelo usuário")


class Estado(TypedDict, total=False):
    pergunta: str
    intencao: str
    codigo: Optional[str]
    resultado: dict          # retorno da ferramenta
    situacao: Optional[str]
    resposta: str


MENSAGENS_FIXAS = {
    "ambigua": "Não identifiquei qual produto você quer consultar. Informe o código no formato PRD-XXX (ex.: PRD-001).",
    "fora_do_escopo": "Só posso responder sobre a disponibilidade e a situação do estoque de produtos. "
                      "Não cadastro, altero nem excluo produtos.",
    "falha_llm": "Não consegui processar a solicitação agora (modelo local indisponível). "
                 "Verifique se o Ollama está rodando e tente novamente.",
}

classificador = llm.with_structured_output(Decisao)


CODIGO_NO_TEXTO = re.compile(r"\bPRD-[0-9]{3}\b", re.IGNORECASE)


def interpretar(estado: Estado) -> Estado:
    try:
        d = classificador.invoke([("system", PROMPT_CLASSIFICACAO), ("human", estado["pergunta"])])
        # O LLM decide a intenção; o código PRD-XXX é extraído por regex (modelos pequenos às vezes o omitem)
        achado = CODIGO_NO_TEXTO.search(estado["pergunta"])
        return {"intencao": d.intencao, "codigo": achado.group(0) if achado else d.codigo}
    except Exception as e:  # Ollama fora do ar, timeout ou saída fora do formato
        print(f"[interpretar] falha no LLM: {e}")
        return {"intencao": "falha_llm"}


def decidir_rota(estado: Estado) -> str:
    if estado["intencao"] == "consultar_produto" and estado.get("codigo"):
        return "consultar_api"
    return "responder_direto"


def consultar_api(estado: Estado) -> Estado:
    resultado = consultar_produto.invoke({"codigo": estado["codigo"]})
    situacao = None
    if resultado["ok"]:
        p = resultado["produto"]
        situacao = classificar_situacao(p["quantidade_atual"], p["estoque_minimo"])
    return {"resultado": resultado, "situacao": situacao}


def responder(estado: Estado) -> Estado:
    r = estado["resultado"]
    if not r["ok"]:
        return {"resposta": f"Não consegui obter os dados do produto {estado['codigo']}: {r['mensagem']}"}

    p = r["produto"]
    resumo = (f"{p['nome']} ({p['codigo']}): {p['quantidade_atual']} {p.get('unidade', '')} em estoque, "
              f"estoque mínimo {p['estoque_minimo']}, situação: {estado['situacao']}.")
    try:
        msg = llm.invoke([
            ("system", PROMPT_SISTEMA),
            ("human", f"Pergunta: {estado['pergunta']}\nDados da API: {resumo}\n"
                      "Responda à pergunta usando apenas esses dados e cite a quantidade atual e o estoque mínimo."),
        ])
        # Garante a quantidade na resposta mesmo se o modelo a omitir (requisito do cenário de teste 1)
        if str(p["quantidade_atual"]) not in msg.content:
            return {"resposta": f"{msg.content}\n{resumo}"}
        return {"resposta": msg.content}
    except Exception as e:
        print(f"[responder] falha no LLM, usando resumo fixo: {e}")
        return {"resposta": resumo}


def responder_direto(estado: Estado) -> Estado:
    if estado["intencao"] == "conceitual":
        try:
            msg = llm.invoke([
                ("system", PROMPT_SISTEMA + "\nResponda de forma conceitual, sem citar produtos nem quantidades."),
                ("human", estado["pergunta"]),
            ])
            return {"resposta": msg.content}
        except Exception as e:
            print(f"[responder_direto] falha no LLM: {e}")
            return {"resposta": MENSAGENS_FIXAS["falha_llm"]}
    # consultar_produto sem código cai aqui e é tratado como ambígua
    return {"resposta": MENSAGENS_FIXAS.get(estado["intencao"], MENSAGENS_FIXAS["ambigua"])}

print("Estado e nós do grafo definidos")

Estado e nós do grafo definidos


## 6. Montagem do grafo

In [ ]:
grafo = StateGraph(Estado)
grafo.add_node("interpretar", interpretar)
grafo.add_node("consultar_api", consultar_api)
grafo.add_node("responder", responder)
grafo.add_node("responder_direto", responder_direto)

grafo.add_edge(START, "interpretar")
grafo.add_conditional_edges("interpretar", decidir_rota, ["consultar_api", "responder_direto"])
grafo.add_edge("consultar_api", "responder")
grafo.add_edge("responder", END)          # condição de encerramento
grafo.add_edge("responder_direto", END)   # condição de encerramento

agente = grafo.compile()

print(agente.get_graph().draw_mermaid())  # diagrama em Mermaid
print("Grafo compilado com sucesso")

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	interpretar(interpretar)
	consultar_api(consultar_api)
	responder(responder)
	responder_direto(responder_direto)
	__end__([<p>__end__</p>]):::last
	__start__ --> interpretar;
	consultar_api --> responder;
	interpretar -.-> consultar_api;
	interpretar -.-> responder_direto;
	responder --> __end__;
	responder_direto --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc

Grafo compilado com sucesso


## 7. Execução

`perguntar()` valida a entrada, executa o grafo com `recursion_limit` (parada garantida mesmo se o fluxo mudar) e mostra o estado final, que serve como evidência.

In [7]:
LIMITE_PERGUNTA = 500


def perguntar(pergunta: str) -> Estado:
    pergunta = (pergunta or "").strip()
    if not pergunta or len(pergunta) > LIMITE_PERGUNTA:
        print(f"Entrada inválida: a pergunta deve ter entre 1 e {LIMITE_PERGUNTA} caracteres.")
        return {}

    estado = agente.invoke({"pergunta": pergunta}, config={"recursion_limit": 10})
    print(f"Pergunta : {pergunta}")
    print(f"Intenção : {estado.get('intencao')} | código: {estado.get('codigo')} | situação: {estado.get('situacao')}")
    print(f"API      : {estado.get('resultado', 'não consultada')}")
    print(f"Resposta : {estado['resposta']}\n")
    return estado

print("Função perguntar pronta para uso")

Função perguntar pronta para uso


### Produto existente (baixo, esgotado e disponível)

In [8]:
perguntar("Qual a situação do estoque do produto PRD-001?")
perguntar("Ainda tem PRD-004 em estoque?")
perguntar("Como está o PRD-002?");

Pergunta : Qual a situação do estoque do produto PRD-001?
Intenção : consultar_produto | código: PRD-001 | situação: baixo
API      : {'ok': True, 'produto': {'codigo': 'PRD-001', 'nome': 'Mouse sem fio', 'categoria': 'Periféricos', 'quantidade_atual': 5, 'estoque_minimo': 10, 'unidade': 'un'}}
Resposta : O produto PRD-001 está em situação de baixo estoque. A quantidade atual é de 5 unidades, enquanto o estoque mínimo é de 10 unidades.

Pergunta : Ainda tem PRD-004 em estoque?
Intenção : consultar_produto | código: PRD-004 | situação: esgotado
API      : {'ok': True, 'produto': {'codigo': 'PRD-004', 'nome': 'Cabo HDMI 2 m', 'categoria': 'Cabos e Adaptadores', 'quantidade_atual': 0, 'estoque_minimo': 20, 'unidade': 'un'}}
Resposta : A quantidade atual de PRD-004 é 0 unidades, com estoque mínimo de 20 unidades.

Pergunta : Como está o PRD-002?
Intenção : consultar_produto | código: PRD-002 | situação: disponível
API      : {'ok': True, 'produto': {'codigo': 'PRD-002', 'nome': 'Teclado US

### Produto inexistente e falhas da API

In [9]:
perguntar("Qual o estoque do PRD-999?")      # 404
perguntar("Consulte o produto PRD-503")       # 503 simulado
perguntar("Quantas unidades tem o ABC-1?");   # código inválido, barrado antes da chamada

Pergunta : Qual o estoque do PRD-999?
Intenção : consultar_produto | código: PRD-999 | situação: None
API      : {'ok': False, 'erro': 'produto_nao_encontrado', 'mensagem': 'Produto não encontrado.'}
Resposta : Não consegui obter os dados do produto PRD-999: Produto não encontrado.

Pergunta : Consulte o produto PRD-503
Intenção : consultar_produto | código: PRD-503 | situação: None
API      : {'ok': False, 'erro': 'servico_indisponivel', 'mensagem': 'Serviço temporariamente indisponível. Tente novamente mais tarde.'}
Resposta : Não consegui obter os dados do produto PRD-503: Serviço temporariamente indisponível. Tente novamente mais tarde.

Pergunta : Quantas unidades tem o ABC-1?
Intenção : consultar_produto | código: ABC-1 | situação: None
API      : {'ok': False, 'erro': 'codigo_invalido', 'mensagem': "O código 'ABC-1' não segue o formato PRD-XXX (ex.: PRD-001)."}
Resposta : Não consegui obter os dados do produto ABC-1: O código 'ABC-1' não segue o formato PRD-XXX (ex.: PRD-001).



### Sem consulta à API (conceitual, ambígua e fora do escopo)

In [10]:
perguntar("O que é estoque mínimo?")
perguntar("Como está o estoque do mouse?")
perguntar("Cadastre o produto PRD-020 com 15 unidades");

Pergunta : O que é estoque mínimo?
Intenção : conceitual | código: None | situação: None
API      : não consultada
Resposta : O estoque mínimo é o nível de estoque abaixo do qual um produto deve ser reabastecido para evitar esgotamento. Quando a quantidade atual atinge ou cai abaixo desse nível, indica que é necessário realizar uma reposição para manter o fluxo de vendas sem interrupções.

Pergunta : Como está o estoque do mouse?
Intenção : ambigua | código: None | situação: None
API      : não consultada
Resposta : Não identifiquei qual produto você quer consultar. Informe o código no formato PRD-XXX (ex.: PRD-001).

Pergunta : Cadastre o produto PRD-020 com 15 unidades
Intenção : fora_do_escopo | código: PRD-020 | situação: None
API      : não consultada
Resposta : Só posso responder sobre a disponibilidade e a situação do estoque de produtos. Não cadastro, altero nem excluo produtos.

